In [ ]:
pip install transformers datasets torch scikit-learn evaluate accelerate

In [ ]:
from transformers import BertTokenizerFast, BertForSequenceClassification

model_checkpoint = 'bert-base-uncased'

# Load the tokenizer
tokenizer = BertTokenizerFast.from_pretrained(model_checkpoint)

# Load the model with 2 labels for binary classification
model = BertForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

print(f"Successfully loaded {model_checkpoint} model and tokenizer.")

In [ ]:
from datasets import load_dataset

# Loading the dataset without trust_remote_code as it's now in parquet format
raw_datasets = load_dataset("stanfordnlp/imdb")
print("Dataset loaded successfully:")
print(raw_datasets)

In [ ]:
# Show an example from the training set only if loading was successful
if 'raw_datasets' in globals():
    print("Example record:")
    display(raw_datasets['train'][0])
else:
    print("Dataset 'raw_datasets' is not available. Please check the previous cell for errors.")

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

# Apply tokenization
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Prepare for PyTorch: remove text and rename label
tokenized_datasets = tokenized_datasets.remove_columns(['text'])
tokenized_datasets = tokenized_datasets.rename_column('label', 'labels')
tokenized_datasets.set_format('torch')

print("Tokenization complete with max_length=128.")
print(f"Column names: {tokenized_datasets['train'].column_names}")

In [ ]:
from torch.utils.data import DataLoader

# Create DataLoaders
batch_size = 16

train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle=True,
    batch_size=batch_size
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"],
    batch_size=batch_size
)

print(f"DataLoaders created with batch_size={batch_size}.")

In [ ]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
import torch

# Move model to device if not already there
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

# 1. Initialize AdamW optimizer with specified hyperparameters
optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

# 2. Calculate training steps
num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
num_warmup_steps = int(0.1 * num_training_steps)

# 3. Initialize the linear warmup scheduler
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

print(f"Device: {device}")
print(f"Total training steps: {num_training_steps}")
print(f"Warmup steps (10%): {num_warmup_steps}")

In [ ]:
import evaluate
import numpy as np
from transformers import TrainingArguments, Trainer

# Load the accuracy metric
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Calculate warmup steps (10% of total)
num_training_steps = 3 * len(train_dataloader)
num_warmup_steps = int(0.1 * num_training_steps)

# Define training arguments with gradient clipping
training_args = TrainingArguments(
    output_dir="bert-imdb-trainer",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=num_warmup_steps,
    max_grad_norm=1.0,  # Gradient clipping added here
    load_best_model_at_end=True,
    logging_steps=100,
    report_to="none"
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

print(f"Trainer initialized with gradient clipping and {num_warmup_steps} warmup steps.")

In [ ]:
from datasets import config
# Workaround for the torchvision.io VideoReader ImportError
config.TORCHVISION_AVAILABLE = False

# Execute the training process
trainer.train()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Get predictions from the trainer
predictions_output = trainer.predict(tokenized_datasets['test'])
predictions = np.argmax(predictions_output.predictions, axis=-1)
labels = predictions_output.label_ids

# 1. Print Accuracy and F1 (Compute Metrics already handled these, but let's show class-wise details)
print("Classification Report:")
print(classification_report(labels, predictions, target_names=['Negative', 'Positive']))

# 2. Plot Confusion Matrix
cm = confusion_matrix(labels, predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
def predict(text):
    """
    Tokenizes input text, performs a forward pass through the model,
    and returns the predicted sentiment label.
    """
    # 1. Tokenize the input text
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    # 2. Model forward pass
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    # 3. Get prediction (argmax on logits)
    logits = outputs.logits
    prediction = torch.argmax(logits, dim=-1).item()

    # Map 0 -> Negative, 1 -> Positive
    return "Positive" if prediction == 1 else "Negative"

# --- Demo on Custom Reviews ---
test_reviews = [
    "I absolutely loved this movie! The acting was top-notch and the plot was gripping from start to finish.",
    "What a waste of time. The script was poorly written and the characters were uninteresting.",
    "It was okay, but I wouldn't watch it again. Some parts were slow, though the ending was decent.",
    "A cinematic masterpiece that redefines the genre. Truly breathtaking.",
    "I disliked every bit of the acting."
]

print(f"--- Custom Inference Demo ---\n")
for review in test_reviews:
    label = predict(review)
    print(f"Review: {review}")
    print(f"Result: {label}\n")